# 05. 도메인 해석 — 무기화학 관점

## 핵심 질문
**XGBoost가 밴드갭을 예측할 때 무엇을 '보고' 있는가?**  
모델의 feature importance를 무기화학·결정장이론 언어로 번역한다.

이 노트북이 프로젝트의 차별점이다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import ast
import xgboost as xgb
from pymatgen.core import Composition
from sklearn.model_selection import train_test_split

df = pd.read_csv('../data/featurized_highk.csv')
model = xgb.XGBRegressor()
model.load_model('../data/xgb_bandgap_model.json')
feature_cols = [c for c in df.columns if c not in ['material_id', 'formula', 'band_gap']]

X_train, X_test, y_train, y_test = train_test_split(df[feature_cols], df['band_gap'], test_size=0.2, random_state=42)
y_pred = model.predict(X_test)
results = df.loc[X_test.index, ['formula', 'band_gap']].copy()
results['predicted'] = y_pred
results['abs_error'] = abs(results['band_gap'] - results['predicted'])
print('로드 완료')

---
## 해석 1. NdValence가 44%를 설명하는 이유 — d 오비탈과 밴드갭

### 무기화학 관점
산화물에서 금속 양이온의 d 오비탈 점유 상태는 밴드갭을 결정하는 가장 중요한 인자다.

- **d⁰ 전이금속 산화물 (NdValence = 0):** Hf⁴⁺(d⁰), Zr⁴⁺(d⁰), Al³⁺(d⁰)  
  → d 오비탈이 비어 있어 valence band(O 2p)와 conduction band(금속 d) 사이 갭이 넓음  
  → **HfO₂: 4.0 eV, ZrO₂: 3.5 eV, Al₂O₃: 5.9 eV**

- **부분 점유 d 전이금속 산화물 (NdValence > 0):** V⁴⁺(d¹), Fe³⁺(d⁵)  
  → d 전자가 갭 안에 상태를 만들거나 conduction band를 낮춤 → 밴드갭 감소

- **avg_dev NdValence (중요도 1위, r = -0.526):** 조성 내 원소들의 d 전자수 **편차**  
  → d⁰ 양이온과 dⁿ 양이온이 섞인 복합 산화물 = 전하 이동 경로 생성 → 갭 감소

In [ ]:
nd_col = 'MagpieData mean NdValence'
avgdev_col = 'MagpieData avg_dev NdValence'

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(df[nd_col], df['band_gap'], alpha=0.3, s=10, color='steelblue')
axes[0].set_xlabel('Mean NdValence (d-orbital electrons)')
axes[0].set_ylabel('Band Gap (eV)')
axes[0].set_title('d-orbital occupancy vs Band Gap')
known = {'HfO2': (0, 4.02), 'ZrO2': (0, 3.53), 'Al2O3': (0, 5.87), 'TiO2': (0, 2.06)}
for name, (nd, bg) in known.items():
    axes[0].annotate(name, (nd + 0.05, bg), fontsize=8, color='red')
    axes[0].scatter([nd], [bg], color='red', s=40, zorder=5)

axes[1].scatter(df[avgdev_col], df['band_gap'], alpha=0.3, s=10, color='darkorange')
axes[1].set_xlabel('avg_dev NdValence (compositional d-electron diversity)')
axes[1].set_ylabel('Band Gap (eV)')
axes[1].set_title('d-electron Diversity vs Band Gap (Top Feature, 24.5%)')

plt.tight_layout()
plt.savefig('../data/interpretation_d_orbital.png', dpi=150)
plt.show()

r1 = df[[nd_col, 'band_gap']].corr().iloc[0,1]
r2 = df[[avgdev_col, 'band_gap']].corr().iloc[0,1]
print(f'NdValence mean vs band_gap    : r = {r1:.3f}')
print(f'NdValence avg_dev vs band_gap : r = {r2:.3f}  ← 1위 feature')

---
## 해석 2. 전기음성도 — 선형 상관은 약하지만 XGBoost는 잡아낸다

### 무기화학 예측 vs 데이터 결과
- **예측:** 전기음성도 차이 클수록 이온결합성 강함 → 밴드갭 증가 (Hf-O > Ti-O)
- **실제:** 선형 상관 r = **0.078** — 거의 무상관

### 이유
산화물 데이터셋이라 모든 소재에 O(전기음성도 3.44)가 포함됨.  
**조성 평균을 구하면 O가 값을 지배**해 양이온의 전기음성도 차이가 묻혀버림.  
→ 단순 평균으로는 신호가 약하지만, XGBoost는 비선형·조건부 패턴으로 이 정보를 활용함 (importance ~1.9%)  
→ **양이온만의 전기음성도를 별도 feature로 추가**하면 모델 성능 개선 가능 (향후 과제)

In [ ]:
en_col = 'MagpieData mean Electronegativity'
r3 = df[[en_col, 'band_gap']].corr().iloc[0,1]
print(f'Mean Electronegativity vs band_gap: r = {r3:.3f}')
print('→ 선형 상관 약함 (O 포함으로 인한 평균 희석 효과)')

plt.figure(figsize=(7, 5))
plt.scatter(df[en_col], df['band_gap'], alpha=0.3, s=10, color='seagreen')
plt.xlabel('Mean Electronegativity (Pauling, O 포함)')
plt.ylabel('Band Gap (eV)')
plt.title(f'Electronegativity vs Band Gap  (r = {r3:.3f})')
plt.tight_layout()
plt.savefig('../data/interpretation_electronegativity.png', dpi=150)
plt.show()

---
## 해석 3. 예측 오류 분석 — 가설과 데이터가 다를 때

### 초기 가설 (틀렸음)
> "란타넘족(f 오비탈)은 Magpie feature로 못 잡으니 예측이 나쁠 것"

### 실제 데이터 결과
| 그룹 | MAE |
|------|-----|
| H 포함 (수화물) | **0.381 eV** (1.4배 ↑) |
| H 미포함 | 0.270 eV |
| 란타넘족 포함 | **0.237 eV** (오히려 더 정확) |
| 란타넘족 미포함 | 0.307 eV |

### 수정된 해석
- **란타넘족 산화물**은 오히려 잘 예측됨. La₂O₃, Y₂O₃ 등은 d⁰ f^n 구성이 일정하고  
  데이터에 충분한 예시가 있어 모델이 패턴을 학습함
- **실제 문제는 H 포함 복합 산화물**: O-H 결합이 전자 구조를 근본적으로 바꾸는데  
  Magpie는 H를 금속 양이온과 동일한 방식으로 처리 → 과대/과소 예측 발생
- **교훈:** 가설은 데이터로 검증되어야 한다. 초기 직관이 틀릴 수 있고,  
  그 틀림 자체가 모델 개선 방향을 알려준다.

In [ ]:
lanthanides = {'La','Ce','Pr','Nd','Pm','Sm','Eu','Gd','Tb','Dy','Ho','Er','Tm','Yb','Lu'}

def has_h(f):
    try: return 'H' in [str(e) for e in Composition(f).elements]
    except: return False

def has_ln(f):
    try: return bool(set(str(e) for e in Composition(f).elements) & lanthanides)
    except: return False

results['has_H'] = results['formula'].apply(has_h)
results['has_Ln'] = results['formula'].apply(has_ln)

mae_h = results[results['has_H']]['abs_error'].mean()
mae_no_h = results[~results['has_H']]['abs_error'].mean()
mae_ln = results[results['has_Ln']]['abs_error'].mean()
mae_no_ln = results[~results['has_Ln']]['abs_error'].mean()

print('=== 그룹별 MAE ===')
print(f'H 포함    : {mae_h:.3f} eV  (n={results["has_H"].sum()}) ← 오류 1.4배')
print(f'H 미포함  : {mae_no_h:.3f} eV  (n={(~results["has_H"]).sum()})')
print(f'란타넘족  : {mae_ln:.3f} eV  (n={results["has_Ln"].sum()}) ← 오히려 더 정확')
print(f'비란타넘족: {mae_no_ln:.3f} eV  (n={(~results["has_Ln"]).sum()})')

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(results[~results['has_H']]['abs_error'], bins=30, alpha=0.6, label='H 미포함', color='steelblue')
axes[0].hist(results[results['has_H']]['abs_error'], bins=20, alpha=0.6, label='H 포함 (수화물)', color='tomato')
axes[0].set_xlabel('Absolute Error (eV)')
axes[0].set_ylabel('Count')
axes[0].set_title('예측 오류: H 포함 vs 미포함')
axes[0].legend()

groups = ['H 미포함', 'H 포함', '란타넘족', '비란타넘족']
maes = [mae_no_h, mae_h, mae_ln, mae_no_ln]
colors = ['steelblue', 'tomato', 'seagreen', 'steelblue']
axes[1].bar(groups, maes, color=colors, alpha=0.8)
axes[1].set_ylabel('MAE (eV)')
axes[1].set_title('그룹별 예측 성능 비교')
axes[1].axhline(y=results['abs_error'].mean(), color='black', linestyle='--', label=f'전체 MAE')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/interpretation_error_analysis.png', dpi=150)
plt.show()

---
## 최종 요약 — 면접 1분 스크립트

```
Materials Project 공개 데이터로 DRAM high-k 유전체 후보 소재의
밴드갭을 XGBoost로 예측했습니다. (MAE 0.29 eV, R² 0.75)

가장 중요한 발견은 feature importance 1위가 NdValence(d 오비탈 전자수)
였다는 점입니다. 이는 d⁰ 전이금속 산화물(HfO₂, ZrO₂)이 갭이 넓은 이유를
결정장이론으로 설명합니다 — d 오비탈이 비어 있으면 O 2p와 금속 d 사이의
에너지 갭이 최대화됩니다.

흥미로운 점은 초기에 란타넘족(f 오비탈)이 예측 오류가 높을 것이라
가설을 세웠지만, 실제 데이터에서는 오히려 더 잘 예측됐습니다.
진짜 문제는 H를 포함한 수화물 계열이었는데, O-H 결합이 전자 구조를
근본적으로 바꾸기 때문입니다. 이 결과는 H-specific descriptor 추가라는
구체적인 모델 개선 방향으로 연결됩니다.
```